# Notebook 01 — Data Generation & Intro to pandas

**Goal**: Generate our synthetic banking dataset and learn the basics of `pandas` — the most important library for ML data work.

**What you'll learn:**
- What a DataFrame is and how to work with it
- How to load, inspect, and slice data
- How to check for missing values
- How to do basic statistics

---

## Step 1 — Generate the data

We use our custom generator to create 1000 fake bank customers.
Each customer has financial attributes: income, spending, savings, credit score, etc.

The generator secretly assigns each customer to one of 4 archetypes:
- **Young Spender** — low income, high spending, low savings
- **Stable Saver** — mid income, low spending, high savings
- **High Earner** — high income, high savings, excellent credit
- **Debt Stressed** — low income, high loans, poor credit

Our ML model doesn't know these labels — it has to *discover* them from the data.

In [1]:
import sys
sys.path.append('..')  # so we can import from src/

import os
import pandas as pd
import numpy as np

from src.data_generator import generate_customers

# Generate 1000 customers
df = generate_customers(n_customers=1000, seed=42)

# Save to disk so other notebooks can load it
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/customers_raw.csv', index=False)

print('Data saved to data/customers_raw.csv')
print(f'Shape: {df.shape}')  # (rows, columns)

Data saved to data/customers_raw.csv
Shape: (1000, 10)


## Step 2 — The DataFrame: your main data structure

A **DataFrame** is a table — rows are observations (customers), columns are features (attributes).
Think of it as a smarter spreadsheet you can program.

`df.head()` shows the first 5 rows.

In [2]:
df.head()

,customer_id,age,annual_income,monthly_spending,savings_balance,credit_score,loan_amount,num_existing_products,months_as_customer,archetype
0,C00001,38,143108.94,5541.53,81516.13,787,17231.22,3,66,high_earner
1,C00002,38,43831.58,2821.91,40560.44,676,5917.93,4,89,stable_saver
2,C00003,23,29655.20,2648.69,813.77,603,43834.20,2,74,debt_stressed
3,C00004,51,123029.45,5656.92,114712.59,788,7707.13,4,83,high_earner
4,C00005,27,36020.48,2050.06,3407.50,635,4295.56,2,12,young_spender


In [3]:
# Shape: (rows, columns)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print()

# Column names
print('Columns:', df.columns.tolist())

Rows: 1000
Columns: 10

Columns: ['customer_id', 'age', 'annual_income', 'monthly_spending', 'savings_balance', 'credit_score', 'loan_amount', 'num_existing_products', 'months_as_customer', 'archetype']


## Step 3 — Data types

Every column has a data type. ML models only understand numbers, so we need to know which columns are:
- `int64` / `float64` → numeric (ready to use)
- `object` → text/categorical (needs encoding later)

`df.info()` gives a quick overview.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            1000 non-null   str    
 1   age                    1000 non-null   int64  
 2   annual_income          1000 non-null   float64
 3   monthly_spending       1000 non-null   float64
 4   savings_balance        973 non-null    float64
 5   credit_score           1000 non-null   int64  
 6   loan_amount            977 non-null    float64
 7   num_existing_products  1000 non-null   int64  
 8   months_as_customer     1000 non-null   int64  
 9   archetype              1000 non-null   str    
dtypes: float64(4), int64(4), str(2)
memory usage: 78.3 KB


## Step 4 — Summary statistics

`df.describe()` gives count, mean, min, max, and percentiles for every numeric column.
This is your first "feel" for the data.

In [5]:
df.describe().round(2)

,age,annual_income,monthly_spending,savings_balance,credit_score,loan_amount,num_existing_products,months_as_customer
count,1000.00,1000.00,1000.00,973.00,1000.00,977.00,1000.00,1000.00
mean,36.44,55845.21,3059.43,28790.69,672.83,13776.25,2.59,54.84
std,10.55,35324.78,1365.43,35708.37,93.73,11826.61,1.42,36.35
min,18.00,15000.00,993.45,0.00,426.00,0.00,1.00,1.00
25%,27.00,30339.26,2192.88,1857.69,599.00,5820.59,2.00,22.00
50%,36.00,41064.95,2630.13,5232.92,675.00,9342.44,2.00,52.00
75%,45.00,66593.59,3256.28,42920.45,756.00,20190.71,4.00,81.00
max,65.00,180666.69,9425.23,158296.16,850.00,54859.78,6.00,199.00


**What to look for:**
- Large gap between mean and max → possible outliers
- min = 0 on `loan_amount` → some customers have no loan (expected)
- Different scales (income is in 100,000s, age is in 20-60) → we'll need to **scale** in notebook 03

## Step 5 — Selecting columns

Two ways to pick columns:
- `df['column_name']` → one column (returns a Series)
- `df[['col1', 'col2']]` → multiple columns (returns a DataFrame)

In [6]:
# Single column — returns a Series (like a list with an index)
df['annual_income'].head()

0    143108.94
1     43831.58
2     29655.20
3    123029.45
4     36020.48
Name: annual_income, dtype: float64

In [7]:
# Multiple columns — returns a DataFrame
df[['customer_id', 'annual_income', 'credit_score']].head()

,customer_id,annual_income,credit_score
0,C00001,143108.94,787
1,C00002,43831.58,676
2,C00003,29655.20,603
3,C00004,123029.45,788
4,C00005,36020.48,635


## Step 6 — Filtering rows

Use conditions inside `[]` to filter, just like a WHERE clause in SQL.

In [8]:
# Customers with annual income > 100,000
high_earners = df[df['annual_income'] > 100_000]
print(f'High earners: {len(high_earners)} customers')
high_earners.head()

High earners: 151 customers


,customer_id,age,annual_income,monthly_spending,savings_balance,credit_score,loan_amount,num_existing_products,months_as_customer,archetype
0,C00001,38,143108.94,5541.53,81516.13,787,17231.22,3,66,high_earner
3,C00004,51,123029.45,5656.92,114712.59,788,7707.13,4,83,high_earner
6,C00007,32,110889.04,2883.75,91081.80,789,NaN,5,102,high_earner
7,C00008,40,119965.18,6683.36,73266.86,810,21774.01,6,67,high_earner
12,C00013,43,130814.86,5574.30,93311.87,779,1018.90,5,115,high_earner


In [9]:
# Multiple conditions: use & (and), | (or)
# Young customers with poor credit
risky_young = df[(df['age'] < 30) & (df['credit_score'] < 600)]
print(f'Young + poor credit: {len(risky_young)} customers')

Young + poor credit: 120 customers


## Step 7 — Missing values

Real data always has missing values. We introduced ~3% missing in `savings_balance` and `loan_amount`.

`df.isnull().sum()` counts missing values per column.

In [10]:
print('Missing values per column:')
print(df.isnull().sum())
print()

# As a percentage
print('Missing %:')
print((df.isnull().sum() / len(df) * 100).round(2))

Missing values per column:
customer_id               0
age                       0
annual_income             0
monthly_spending          0
savings_balance          27
credit_score              0
loan_amount              23
num_existing_products     0
months_as_customer        0
archetype                 0
dtype: int64

Missing %:
customer_id              0.0
age                      0.0
annual_income            0.0
monthly_spending         0.0
savings_balance          2.7
credit_score             0.0
loan_amount              2.3
num_existing_products    0.0
months_as_customer       0.0
archetype                0.0
dtype: float64


We'll handle these missing values properly in **notebook 03 (Preprocessing)**.

## Step 8 — Groupby: aggregate by category

`groupby` groups rows by a column value, then applies an aggregation.
Think of it as: "for each archetype, what's the average income?"

In [11]:
# Average income and savings per archetype
# (We're peeking at archetypes here just to understand the data — the ML model won't see this)
df.groupby('archetype')[['annual_income', 'savings_balance', 'credit_score', 'loan_amount']].mean().round(0)

,annual_income,savings_balance,credit_score,loan_amount
archetype,,,,
debt_stressed,27937.0,1224.0,555.0,33014.0
high_earner,117881.0,87883.0,788.0,15566.0
stable_saver,57494.0,34620.0,730.0,6053.0
young_spender,31834.0,2528.0,618.0,7857.0


In [12]:
# How many customers per archetype?
df['archetype'].value_counts()

archetype
young_spender    308
stable_saver     295
high_earner      200
debt_stressed    197
Name: count, dtype: int64

## Step 9 — Adding a computed column

Feature engineering starts here — creating new columns from existing ones.

In [13]:
# Spending ratio: monthly spending as % of monthly income
df['spending_ratio'] = (df['monthly_spending'] / (df['annual_income'] / 12)).round(4)

# Savings ratio: savings as % of annual income
df['savings_ratio'] = (df['savings_balance'] / df['annual_income']).round(4)

print('New columns added: spending_ratio, savings_ratio')
df[['annual_income', 'monthly_spending', 'spending_ratio', 'savings_balance', 'savings_ratio']].head()

New columns added: spending_ratio, savings_ratio


,annual_income,monthly_spending,spending_ratio,savings_balance,savings_ratio
0,143108.94,5541.53,0.4647,81516.13,0.5696
1,43831.58,2821.91,0.7726,40560.44,0.9254
2,29655.20,2648.69,1.0718,813.77,0.0274
3,123029.45,5656.92,0.5518,114712.59,0.9324
4,36020.48,2050.06,0.6830,3407.50,0.0946


## Checkpoint — What you learned

| Concept | pandas method |
|---|---|
| Load/view data | `df.head()`, `df.shape` |
| Data types | `df.info()` |
| Summary stats | `df.describe()` |
| Select columns | `df['col']`, `df[['a','b']]` |
| Filter rows | `df[condition]` |
| Missing values | `df.isnull().sum()` |
| Group & aggregate | `df.groupby().mean()` |
| New column | `df['new'] = expression` |

**Next up: Notebook 02 — Exploratory Data Analysis (EDA)**
We'll visualize distributions, spot outliers, and understand feature correlations.